# TOPIC vs RAG 비교 노트북

질문 하나를 넣으면 TOPIC 분류 경로와 10-Stage RAG 파이프라인(Pattern Card -> Query Plan -> Fragment -> 단계별 SQL 생성 -> Composer)이 각각 어떤 topic/pattern/fragment/rule을 골랐는지, 그리고 최종 조립된 SQL을 나란히 보여준다.

실제 조회/분류/SQL 생성은 전부 `server/rag-poc/inspectQuery.mjs`(Node)가 수행한다 — 이 노트북은 그 스크립트를 서브프로세스로 호출해 JSON을 받아 표로 정리하는 얇은 뷰어일 뿐이다.

In [1]:
import json
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

# inspectQuery.mjs가 dotenv로 .env를 읽는데, dotenv는 process.cwd() 기준으로 .env를 찾는다.
# .env는 louis/dashboard/에 있으므로 반드시 그 디렉터리를 cwd로 놓고 node를 실행해야 한다.
DASHBOARD_DIR = Path.cwd().parent.parent  # server/rag-poc -> server -> dashboard
assert (DASHBOARD_DIR / '.env').exists(), f"{DASHBOARD_DIR}/.env 를 못 찾음 — 노트북을 server/rag-poc/에서 실행 중인지 확인"


def run_query(query: str, execute: bool = True) -> dict:
    """inspectQuery.mjs를 호출해 TOPIC/RAG 양쪽의 라우팅 로그 + 생성 SQL(+ 실행 결과)을 받아온다."""
    cmd = ['node', 'server/rag-poc/inspectQuery.mjs', query]
    if execute:
        cmd.append('--execute')
    proc = subprocess.run(cmd, cwd=DASHBOARD_DIR, capture_output=True, text=True, encoding='utf-8')
    if proc.returncode != 0:
        raise RuntimeError(f"inspectQuery.mjs 실패 (exit={proc.returncode})\nstdout: {proc.stdout}\nstderr: {proc.stderr}")
    return json.loads(proc.stdout)

## 질문 입력

`QUERY`를 바꾸고 아래 셀부터 다시 실행하면 된다. `EXECUTE=False`로 하면 사용자용 최종 SQL 실행은 건너뛴다 — 단, Stage 9b의 TOP-1 검증 호출은 파이프라인 내부에서 항상 수행된다(라우팅/생성 SQL 정확도 확인이 목적이면 그걸로 충분히 빠르다).

In [2]:
QUERY = "렉서스 강남 딜러의 2026년 4월 계약 건수를 보여줘"  # GOLD=280 검증된 케이스
EXECUTE = True

result = run_query(QUERY, execute=EXECUTE)
print(f"질문: {result['query']}")

질문: 렉서스 강남 딜러의 2026년 4월 계약 건수를 보여줘


## 1. TOPIC 경로 — 어떤 topic을 골랐는지

In [3]:
t = result['topic']
display(Markdown(f"**선택된 topic**: `{t['topic']}`  \n**routing 문서 길이**: {t.get('routingNotesLength', 0)}자  \n**지연시간**: {t['latencyMs']}ms"))

display(Markdown("### 이 topic 번들에 포함된 테이블"))
display(pd.DataFrame(t['tables']))

**선택된 topic**: `계약`  
**routing 문서 길이**: 12875자  
**지연시간**: 4975ms

### 이 topic 번들에 포함된 테이블

,db,id
0,KPI_W,FCT_CONTRACT_KTWS
1,KPI_W,FCT_LEAD
2,KPI_W,FCT_ACTIVITY_v2
3,KPI_W,FCT_CRM_TARGET_M
4,KPI_W,DIM_MNG_USER
5,KPI_W,DIM_MNG_DEALER
6,KPI_W,DIM_VEHIC_SPEC
7,KPI_W,DIM_VEHIC_SPEC_VAR
8,KPI_W,DIM_VEHIC_SPEC_MDL
9,KPI_W,DIM_PMA_ORDER


## 2. RAG 경로 — Stage별로 어떤 것을 골랐는지

In [4]:
r = result['rag']
display(Markdown(f"**지연시간**: {r['latencyMs']}ms"))

display(Markdown("### Stage 0 — 구조화된 질문"))
display(r.get('stage0_structured'))

display(Markdown("### Stage 1 — 스키마(테이블) 검색 결과 (유사도순)"))
display(pd.DataFrame(r['stage1_tables']))

display(Markdown("### Stage 2 — Pattern Card 검색 결과"))
display(pd.DataFrame(r['stage2_patterns']) if r['stage2_patterns'] else "(없음 — MIN_SCORE 미달)")

display(Markdown("### Stage 3 — 선택된 Pattern Card + Query Plan"))
if r.get('stage3_plan'):
    display(Markdown(f"선택된 pattern: `{r['stage3_plan']['pattern_id']}`"))
    display(pd.DataFrame(r['stage3_plan']['steps']))
else:
    display(Markdown(f"⚠️ Pattern Card 선택 실패: {r.get('error')}"))

display(Markdown("### Stage 4 — 해석된 Fragment"))
display(pd.DataFrame(r.get('stage4_fragments', [])) if r.get('stage4_fragments') else "(없음)")

display(Markdown("### Stage 5 — 누락 테이블 백필"))
missing = r.get('stage5_backfill', {}).get('missingTables', [])
display(pd.DataFrame(missing) if missing else "(누락 없음)")

display(Markdown("### Stage 6 — 참조된 업무 규칙"))
display(pd.DataFrame(r.get('stage6_rules', [])) if r.get('stage6_rules') else "(없음)")

display(Markdown("### Stage 6 — 용어 사전 검색"))
display(pd.DataFrame(r.get('stage6_glossary', [])) if r.get('stage6_glossary') else "(없음)")

display(Markdown("### 최종적으로 사용된 테이블"))
display(pd.DataFrame(r.get('tables', [])))

**지연시간**: 6817ms

### Stage 0 — 구조화된 질문

{'intent': ['show'],
 'metrics': ['계약 건수'],
 'dimensions': [],
 'time_range': '2026년 4월',
 'filters': [],
 'operations': ['filter'],
 'estimated_complexity': 'simple',
 'entities': {'dealer': '렉서스 강남'}}

### Stage 1 — 스키마(테이블) 검색 결과 (유사도순)

,db,id,ko,score
0,KPI_W,FCT_CONTRACT_KTWS,계약 실적,0.473794
1,KPI_W,FCT_SALES_TARGET_DAILY,일별 계약/출고 목표,0.418837
2,KPI_W,FCT_ACTIVITY_v2,영업활동 실적,0.379402
3,KPI_W,DIM_PMA_ORDER,PMA in/out 순서,0.361780
4,KPI_W,FCT_LEAD,영업기회(리드) 실적,0.350247


...

,rule_id,term
0,br_yearmonth_scope,조회 연월 필터
1,br_active_staff,재직 SC만 포함
2,br_dealer_scope,대상 딜러 한정
3,br_exclude_front_sc,창구SC 제외
4,br_exclude_staff_names,특정 조직 제외
5,br_exclude_test_users,테스트/특수 계정 제외
6,br_activity_type_scope,활동유형 대분류 제한
7,br_qualified_lead_def,자격 리드(qualified lead) 정의


### Stage 6 — 용어 사전 검색

,id,term,definition,score
0,g03,활동 건수 vs 실적 건수,FCT_ACTIVITY_v2의 cnt(전체 활동 건수)와 actual_cnt(KPI...,0.437833
1,g17,contract_ratio,FCT_HBOARD_MEETING.contract_ratio — 해피보드 미팅에서 ...,0.428485
2,g08,ac_trgt vs co_trgt,"FCT_SALES_TARGET_DAILY의 ac_trgt=출고(딜리버리) 목표, c...",0.403025


### 최종적으로 사용된 테이블

,db,id
0,KPI_W,DIM_CALENDAR_KTWS
1,KPI_W,DIM_MNG_USER
2,KPI_W,DIM_MNG_DEALER
3,KPI_W,FCT_ACTIVITY_v2
4,KPI_W,DIM_CRM_ACT_TYPE
5,KPI_W,FCT_LEAD
6,KPI_W,FCT_CONTRACT_KTWS


## 3. Stage 7 — 단계별로 생성된 SQL Fragment

In [5]:
for step in result['rag'].get('stage7_steps', []):
    display(Markdown(f"**{step['cte_name']}**"))
    display(Markdown(f"```sql\n{step['sql_body']}\n```"))

**mtd_dates**

```sql
SELECT [Date] FROM ktws.DIM_CALENDAR_KTWS WHERE [Date] >= '2026-04-01' AND [Date] <= '2026-04-30'
```

**filtered_users_dealer**

```sql
SELECT D.sc_key FROM ktws.DIM_MNG_USER AS D
    INNER JOIN ktws.DIM_MNG_DEALER AS E ON D.dealer_key = E.dealer_key
    WHERE D.active_yn = '재직' AND E.dealer_nm = '렉서스 강남'
      AND D.facade_sc_yn <> '창구SC'
      AND D.name NOT IN ('고객지원팀', 'TOYOTA YM')
      AND D.user_id NOT IN ('DM5103', 'DM9999', 'JB30016', 'KMKS176', 'KMKS999', 'JB30067')
```

**activity_leads_mtd**

```sql
SELECT DISTINCT ACT.lead_key
FROM ktws.FCT_ACTIVITY_v2 AS ACT
INNER JOIN mtd_dates AS DT ON ACT.act_dt_fr = DT.[Date]
INNER JOIN ktws.DIM_CRM_ACT_TYPE AS B ON ACT.tp_key = B.tp_key
WHERE B.tp_grp_1 IN ('관계형성', '기회창출')
  AND (ACT.act_result <> '부재중' OR ACT.act_result IS NULL)
  AND ACT.lead_key IS NOT NULL
```

**lead_pool_mtd**

```sql
SELECT DISTINCT G.lead_key
FROM ktws.FCT_LEAD AS G
INNER JOIN mtd_dates AS DT ON G.lead_reg_dt = DT.[Date]
INNER JOIN activity_leads_mtd AS A ON G.lead_key = A.lead_key
WHERE (G.close_dt > '2026-04-30' OR G.close_dt IS NULL OR G.last_retail_sales_dt IS NOT NULL)
```

**contract_scalar**

```sql
SELECT COALESCE(SUM(I.cnt), 0) AS contract_cnt
FROM ktws.FCT_CONTRACT_KTWS AS I
INNER JOIN mtd_dates AS DT ON I.contract_dt = DT.[Date]
INNER JOIN filtered_users_dealer AS U ON I.cn_sc_key = U.sc_key
INNER JOIN lead_pool_mtd AS L ON I.lead_key = L.lead_key
```

## 4. 생성된 SQL 비교 (TOPIC 단일 생성 vs RAG Composer 조립 결과)

In [6]:
def show_sql(label, node):
    if node.get('error'):
        display(Markdown(f"**{label}**: ⚠️ {node['error']}"))
        return
    display(Markdown(f"**{label}** (db=`{node.get('sqlDb')}`)"))
    display(Markdown(f"```sql\n{node['sql']}\n```"))

show_sql('TOPIC', result['topic'])
show_sql('RAG', result['rag'])

v = result['rag'].get('validation')
if v:
    display(Markdown(f"**RAG 검증**: liveCheckPassed=`{v.get('liveCheckPassed')}`, repairAttempts=`{v.get('repairAttempts')}`, structuralWarnings=`{v.get('structuralWarnings')}`"))

**TOPIC** (db=`KPI_W`)

```sql
WITH qualified_lead AS (  SELECT DISTINCT l.lead_key  FROM ktws.FCT_LEAD AS l  WHERE ( l.close_dt > '2026-04-30' OR l.close_dt IS NULL OR l.last_retail_sales_dt IS NOT NULL )    AND EXISTS (      SELECT 1 FROM ktws.FCT_ACTIVITY_v2 AS a          INNER JOIN ktws.DIM_CRM_ACT_TYPE AS at2 ON a.tp_key = at2.tp_key      WHERE a.lead_key = l.lead_key        AND at2.tp_grp_1 IN ('관계형성', '기회창출')        AND a.act_result <> '부재중'        AND a.lead_key IS NOT NULL    ) ) SELECT TOP 1 COALESCE(SUM(f.cnt), 0) AS contract_cnt FROM ktws.FCT_CONTRACT_KTWS AS f     INNER JOIN ktws.DIM_CALENDAR_KTWS AS c ON f.contract_dt = c.[Date]     INNER JOIN ktws.DIM_MNG_USER AS u ON f.cn_sc_key = u.sc_key     INNER JOIN ktws.DIM_MNG_DEALER AS d ON u.dealer_key = d.dealer_key     INNER JOIN qualified_lead AS ql ON f.lead_key = ql.lead_key WHERE c.[Year] = 2026 AND c.[MonthNumber] = 4     AND d.dealer_nm = '렉서스 강남'     AND (u.facade_sc_yn <> '창구SC' OR u.facade_sc_yn IS NULL)     AND (u.[name] NOT IN ('고객지원팀', 'TOYOTA YM') OR u.[name] IS NULL)
```

**RAG** (db=`KPI_W`)

```sql
WITH mtd_dates AS (
SELECT [Date] FROM ktws.DIM_CALENDAR_KTWS WHERE [Date] >= '2026-04-01' AND [Date] <= '2026-04-30'
),
filtered_users_dealer AS (
SELECT D.sc_key FROM ktws.DIM_MNG_USER AS D
    INNER JOIN ktws.DIM_MNG_DEALER AS E ON D.dealer_key = E.dealer_key
    WHERE D.active_yn = '재직' AND E.dealer_nm = '렉서스 강남'
      AND D.facade_sc_yn <> '창구SC'
      AND D.name NOT IN ('고객지원팀', 'TOYOTA YM')
      AND D.user_id NOT IN ('DM5103', 'DM9999', 'JB30016', 'KMKS176', 'KMKS999', 'JB30067')
),
activity_leads_mtd AS (
SELECT DISTINCT ACT.lead_key
FROM ktws.FCT_ACTIVITY_v2 AS ACT
INNER JOIN mtd_dates AS DT ON ACT.act_dt_fr = DT.[Date]
INNER JOIN ktws.DIM_CRM_ACT_TYPE AS B ON ACT.tp_key = B.tp_key
WHERE B.tp_grp_1 IN ('관계형성', '기회창출')
  AND (ACT.act_result <> '부재중' OR ACT.act_result IS NULL)
  AND ACT.lead_key IS NOT NULL
),
lead_pool_mtd AS (
SELECT DISTINCT G.lead_key
FROM ktws.FCT_LEAD AS G
INNER JOIN mtd_dates AS DT ON G.lead_reg_dt = DT.[Date]
INNER JOIN activity_leads_mtd AS A ON G.lead_key = A.lead_key
WHERE (G.close_dt > '2026-04-30' OR G.close_dt IS NULL OR G.last_retail_sales_dt IS NOT NULL)
)
SELECT TOP 100 COALESCE(SUM(I.cnt), 0) AS contract_cnt
FROM ktws.FCT_CONTRACT_KTWS AS I
INNER JOIN mtd_dates AS DT ON I.contract_dt = DT.[Date]
INNER JOIN filtered_users_dealer AS U ON I.cn_sc_key = U.sc_key
INNER JOIN lead_pool_mtd AS L ON I.lead_key = L.lead_key
```

**RAG 검증**: liveCheckPassed=`True`, repairAttempts=`0`, structuralWarnings=`{'unknownTables': [], 'unknownColumns': []}`

## 5. 실행 결과 비교 (`EXECUTE=True`일 때만)

In [7]:
def show_exec(label, node):
    if not EXECUTE:
        print(f"{label}: EXECUTE=False라 실행 안 함")
        return
    if node.get('execError'):
        display(Markdown(f"**{label}** 실행 오류: {node['execError']}"))
        return
    rows = node.get('execRows')
    if rows is None:
        print(f"{label}: 실행된 SQL 없음")
        return
    display(Markdown(f"**{label}** — {node.get('execRowCount', len(rows))}행 반환"))
    display(pd.DataFrame(rows))

show_exec('TOPIC', result['topic'])
show_exec('RAG', result['rag'])

**TOPIC** — 1행 반환

,contract_cnt
0,324


**RAG** — 1행 반환

,contract_cnt
0,280


## 6. (선택) 여러 질문 한 번에 돌려서 비교표 만들기

In [ ]:
QUERIES = [
    "특정 딜러의 이번달 활동 건수를 보여줘",
    "특정 딜러의 이번달 영업기회 건수를 보여줘",
    "특정 딜러의 이번달 계약 건수를 보여줘",
]

rows = []
results_by_query = {}
for q in QUERIES:
    res = run_query(q, execute=True)
    results_by_query[q] = res
    t, r = res['topic'], res['rag']
    rows.append({
        'query': q,
        'topic_selected': t['topic'],
        'topic_error': t.get('error'),
        'topic_rows': t.get('execRowCount'),
        'rag_pattern': r.get('stage3_plan', {}).get('pattern_id') if r.get('stage3_plan') else None,
        'rag_repair_attempts': r.get('validation', {}).get('repairAttempts'),
        'rag_error': r.get('error'),
        'rag_rows': r.get('execRowCount'),
    })

pd.DataFrame(rows)